# Portfolio Risk Analysis

This notebook analyzes a simple long-only portfolio using daily market data, Monte Carlo simulation, Markowitz optimization, the Capital Market Line, and VaR/CVaR risk metrics.

The reusable logic is implemented in the `src/portfolio_risk/` package. This notebook keeps the analysis narrative clean and avoids hardcoding key financial outputs.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from portfolio_risk.data import download_prices, download_risk_free_rate
from portfolio_risk.markowitz import capital_market_line, efficient_frontier, sharpe_ratio, tangency_portfolio
from portfolio_risk.monte_carlo import simulate_multivariate_portfolio, simulate_univariate_portfolio
from portfolio_risk.plotting import plot_capital_market_line, plot_efficient_frontier, plot_return_distribution
from portfolio_risk.portfolios import portfolio_return_series, random_long_only_portfolios
from portfolio_risk.returns import annualize_covariance, annualize_return, annualize_volatility, deannualize_return, simple_returns
from portfolio_risk.risk_metrics import historical_cvar, historical_var, parametric_cvar, parametric_var

pd.options.display.float_format = "{:.6f}".format


## Configuration

The analysis uses the same four assets as the original notebook. All portfolio names and code are written in English.


In [ ]:
ASSETS = ["KO", "MSFT", "QQQ", "SPY"]
PERIOD = "5y"
PRICE_FIELD = "Close"
TRADING_DAYS = 252
SEED = 42
INITIAL_CAPITAL = 100.0
CONFIDENCE_LEVEL = 0.95

example_weights = {
    "equal_weight": np.array([0.25, 0.25, 0.25, 0.25]),
    "growth_tilt": np.array([0.0, 1 / 3, 1 / 3, 1 / 3]),
    "defensive_tilt": np.array([0.50, 1 / 6, 1 / 6, 1 / 6]),
}


## Data Download And Returns

Prices are downloaded with `yfinance`. The analysis uses simple daily returns and keeps daily quantities separate from annualized quantities.


In [ ]:
prices = download_prices(ASSETS, period=PERIOD, price_field=PRICE_FIELD)
daily_returns = simple_returns(prices).dropna()

daily_mean_returns = daily_returns.mean().reindex(ASSETS)
daily_covariance = daily_returns.cov().reindex(index=ASSETS, columns=ASSETS)
annual_mean_returns = annualize_return(daily_mean_returns)
annual_covariance = annualize_covariance(daily_covariance)
annual_volatilities = annualize_volatility(np.sqrt(np.diag(daily_covariance)))

summary = pd.DataFrame({
    "daily_mean_return": daily_mean_returns,
    "daily_volatility": np.sqrt(np.diag(daily_covariance)),
    "annual_mean_return": annual_mean_returns,
    "annual_volatility": annual_volatilities,
})
summary


## Example Portfolios

Three simple long-only portfolios are constructed to compare daily return behavior before running simulations and optimization.


In [ ]:
portfolio_returns = pd.DataFrame(index=daily_returns.index)
for name, weights in example_weights.items():
    portfolio_returns[name] = portfolio_return_series(daily_returns, weights, columns=ASSETS)

portfolio_returns.describe()


## Monte Carlo Simulation

The first simulation samples returns from the equal-weight portfolio distribution. The second simulation samples asset returns jointly using the covariance matrix.


In [ ]:
equal_weight_returns = portfolio_returns["equal_weight"]

univariate_terminal_capital = simulate_univariate_portfolio(
    mean_daily_return=equal_weight_returns.mean(),
    daily_volatility=equal_weight_returns.std(),
    initial_capital=INITIAL_CAPITAL,
    days=TRADING_DAYS,
    n_simulations=1_000,
    seed=SEED,
)

multivariate_terminal_capital = simulate_multivariate_portfolio(
    mean_daily_returns=daily_mean_returns.to_numpy(),
    daily_covariance=daily_covariance.to_numpy(),
    weights=example_weights["equal_weight"],
    initial_capital=INITIAL_CAPITAL,
    days=TRADING_DAYS,
    n_simulations=1_000,
    seed=SEED,
)

pd.DataFrame({
    "univariate": univariate_terminal_capital.describe(),
    "multivariate": multivariate_terminal_capital.describe(),
})


## Random Portfolios And Efficient Frontier

Random long-only portfolios are generated from the daily expected returns and covariance matrix. The Markowitz frontier is computed by minimizing volatility for target returns.


In [ ]:
random_portfolios = random_long_only_portfolios(
    expected_returns=daily_mean_returns,
    covariance=daily_covariance,
    asset_names=ASSETS,
    n_portfolios=10_000,
    seed=SEED,
)

frontier_daily = efficient_frontier(
    expected_returns=daily_mean_returns.to_numpy(),
    covariance=daily_covariance.to_numpy(),
    n_points=200,
)

random_portfolios.sort_values("sharpe_ratio", ascending=False).head()


In [ ]:
ax = plot_efficient_frontier(random_portfolios, frontier_daily)
ax.set_title("Daily Efficient Frontier")
plt.show()


## Risk-Free Rate, Tangency Portfolio, And Capital Market Line

The risk-free rate is downloaded from `^IRX`, aligned to the returns index, and treated as an annualized decimal yield. The tangency portfolio is optimized using annualized returns and covariance.


In [ ]:
risk_free_annual_series = download_risk_free_rate(period=PERIOD, target_index=daily_returns.index)
risk_free_rate_annual = risk_free_annual_series.mean()
risk_free_rate_daily = deannualize_return(risk_free_rate_annual)

tangency_result = tangency_portfolio(
    expected_returns=annual_mean_returns.to_numpy(),
    covariance=annual_covariance.to_numpy(),
    risk_free_rate=risk_free_rate_annual,
)

tangency_weights = pd.Series(tangency_result.x, index=ASSETS, name="tangency_weight")
tangency_return_annual = float(tangency_weights @ annual_mean_returns)
tangency_volatility_annual = float(np.sqrt(tangency_weights.to_numpy() @ annual_covariance.to_numpy() @ tangency_weights.to_numpy()))
tangency_sharpe = sharpe_ratio(tangency_return_annual, tangency_volatility_annual, risk_free_rate_annual)

pd.DataFrame({
    "tangency_weight": tangency_weights,
})


In [ ]:
pd.Series({
    "risk_free_rate_annual": risk_free_rate_annual,
    "risk_free_rate_daily": risk_free_rate_daily,
    "tangency_return_annual": tangency_return_annual,
    "tangency_volatility_annual": tangency_volatility_annual,
    "tangency_sharpe": tangency_sharpe,
})


In [ ]:
cml = capital_market_line(
    risk_free_rate_annual=risk_free_rate_annual,
    tangency_return_annual=tangency_return_annual,
    tangency_volatility_annual=tangency_volatility_annual,
)

annual_portfolios = random_portfolios.copy()
annual_portfolios["annual_return"] = annualize_return(annual_portfolios["daily_return"])
annual_portfolios["annual_volatility"] = annualize_volatility(annual_portfolios["daily_volatility"])

frontier_annual = frontier_daily.copy()
frontier_annual["annual_return"] = annualize_return(frontier_annual["daily_return"])
frontier_annual["annual_volatility"] = annualize_volatility(frontier_annual["daily_volatility"])

ax = plot_capital_market_line(
    annual_portfolios=annual_portfolios,
    annual_frontier=frontier_annual,
    cml=cml,
    tangency_return_annual=tangency_return_annual,
    tangency_volatility_annual=tangency_volatility_annual,
    risk_free_rate_annual=risk_free_rate_annual,
)
ax.set_title("Capital Market Line")
plt.show()


## VaR And CVaR

VaR and CVaR are reported as positive daily losses. The underlying return quantiles and tail means are shown separately so the sign convention is transparent.


In [ ]:
tangency_daily_returns = portfolio_return_series(daily_returns, tangency_weights.to_numpy(), columns=ASSETS)
tangency_return_daily = deannualize_return(tangency_return_annual)
tangency_volatility_daily = tangency_volatility_annual / np.sqrt(TRADING_DAYS)

historical_var_loss, historical_var_return_quantile = historical_var(tangency_daily_returns, CONFIDENCE_LEVEL)
parametric_var_loss, parametric_var_return_quantile = parametric_var(
    tangency_return_daily,
    tangency_volatility_daily,
    CONFIDENCE_LEVEL,
)
historical_cvar_loss, historical_cvar_tail_mean = historical_cvar(tangency_daily_returns, CONFIDENCE_LEVEL)
parametric_cvar_loss, parametric_cvar_tail_mean = parametric_cvar(
    tangency_return_daily,
    tangency_volatility_daily,
    CONFIDENCE_LEVEL,
)

risk_metrics = pd.Series({
    "historical_var_loss": historical_var_loss,
    "historical_var_return_quantile": historical_var_return_quantile,
    "parametric_var_loss": parametric_var_loss,
    "parametric_var_return_quantile": parametric_var_return_quantile,
    "historical_cvar_loss": historical_cvar_loss,
    "historical_cvar_tail_mean": historical_cvar_tail_mean,
    "parametric_cvar_loss": parametric_cvar_loss,
    "parametric_cvar_tail_mean": parametric_cvar_tail_mean,
})
risk_metrics


In [ ]:
ax = plot_return_distribution(
    tangency_daily_returns,
    historical_var_quantile=historical_var_return_quantile,
    historical_cvar_tail_mean=historical_cvar_tail_mean,
)
ax.set_title("Tangency Portfolio Daily Return Risk")
plt.show()


## Interpretation

The tangency portfolio, Capital Market Line, VaR, and CVaR are all computed from the downloaded data. Results can change when the market data changes.

The key modeling limitations are the long-only constraint, sample-based estimates, normality assumptions in Monte Carlo and parametric risk metrics, and the absence of costs, taxes, liquidity constraints, and rebalancing rules.
